In [2]:
import plotly.graph_objects as go
import numpy as np
import plotly
from IPython.display import display, HTML
from scipy.optimize import minimize, LinearConstraint
import pandas as pd
from dataclasses import dataclass, field
from tqdm import tqdm
from plotly.subplots import make_subplots
import plotly.express as px
from abc import ABC, abstractmethod


In [3]:
discounts = pd.read_excel('discounts.xlsx', index_col='date').dropna()
panel_data = pd.read_excel('macrodata.xlsx',  index_col='meeting_date').dropna()

In [4]:
# Jump size
ref_date = discounts.index[1000]
dt = 1/360
jump_size = 25/10_000
possible_jumps = np.arange(-6*jump_size, 5*jump_size, jump_size)
meeting_dates = pd.to_datetime(panel_data.index)
absolute_meeting_times = np.asarray((meeting_dates - ref_date).days / 360.0)

In [5]:
@dataclass(frozen=True)
class Curve:
    tenors: np.ndarray
    dfs: np.ndarray

    def discount(self, t):
        if t == 0.0:
            return 1.0
        return np.interp(t, self.tenors, self.dfs)

    def fwd_rate(self, t, T):
        df1 = self.discount(t)
        df2 = self.discount(T)
        return -np.log(df2/df1)/(T-t)

    def n_nodes(self) -> int:
        return len(self.tenors)

    def get_tenors(self) -> np.ndarray:
        return self.tenors

    def get_dfs(self) -> np.ndarray:
        return self.dfs

curve = Curve(discounts.columns, discounts.loc[ref_date])
r0 = (1/curve.discount(dt) - 1) / dt

In [6]:
tt = np.arange(0.0, 5.0 + 0.5, 0.5)
fwd_rates = np.array([curve.fwd_rate(t, t + dt) for t in tt])

fig = go.Figure()
fig.add_trace(go.Scatter(x=tt, y=fwd_rates * 100, mode='lines+markers', name='Fwd Rates (%)'))
fig.update_layout(title='Forward Rates Implied by Discount Curve', xaxis_title='Date', yaxis_title='Forward Rate (%)')

In [7]:
@dataclass(frozen=True)
class JumpModelState:
    """
    Represents the state of a jump model.
    """
    r0: float
    curve: Curve
    absolute_meeting_times: np.ndarray
    possible_jumps: np.ndarray
    jump_size: float

    meetings_visibility: int = field(default=1)
    meeting_freq: float = field(default=0.25)

    def relevant_meeting_times(self, t: float, T: float) -> np.ndarray:
        """
            Returns meeting times between t and T, adding synthetic meetings at frequency meeting_freq if needed.
        """
        mts = self.absolute_meeting_times
        mts = mts[(mts > t) & (mts <= T) & (mts <= self.meetings_visibility)]
        if mts.size == 0:
            return mts
        if mts.max() < T:  # we need to fill with synthetic meeting at T
            synth_meetings = np.arange(
                mts.max() + self.meeting_freq, T + self.meeting_freq, self.meeting_freq)
            synth_meetings = synth_meetings[synth_meetings <= T]
            mts = np.concatenate((mts, synth_meetings))
        return mts

    def curve_implied_meeting_dfs(self) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """ 
            Returns deflacted discount factors by the current base rate, along with 
            start times, end times, and time deltas for each meeting segment.
        """
        end_times = self.relevant_meeting_times(
            0.0, self.curve.get_tenors().max())
        if end_times.size == 0:
            raise ValueError("JumpModelState: No meetings within horizon.")
        start_times = np.concatenate(([0.0], end_times[:-1]))

        vec_discount = np.vectorize(self.curve.discount)
        df_T = vec_discount(end_times)
        df_t = vec_discount(start_times)
        time_deltas = end_times - start_times
        per_seg_obs = (df_T / df_t)
        dfs = per_seg_obs / np.exp(-self.r0 * time_deltas)
        return dfs, start_times, end_times, time_deltas


class ProbabilityModel(ABC):
    def __init__(self):
        self._probs: np.ndarray | None = None

    @property
    def probs(self) -> np.ndarray:
        if self._probs is None:
            raise ValueError("ProbabilityModel: probs not set.")
        return self._probs

    @probs.setter
    def probs(self, value: np.ndarray) -> None:
        self._probs = value

    @abstractmethod
    def fit(self, *args, **kwargs) -> None:
        """Fit the model and set probabilities."""
        pass

    def is_ready(self) -> bool:
        return self._probs is not None

    def clear(self) -> None:
        self._probs = None


class JumpModel:
    def __init__(self, r0: float, curve: Curve, absolute_meeting_times: np.ndarray,
                 possible_jumps: np.ndarray, jump_size: float,
                 probability_model: ProbabilityModel):
        self.state = JumpModelState(
            r0, curve, absolute_meeting_times, possible_jumps, jump_size)
        self.probability_model = probability_model

    @staticmethod
    def from_state(state: JumpModelState, probability_model: ProbabilityModel):
        jm = JumpModel(state.r0, state.curve, state.absolute_meeting_times,
                       state.possible_jumps, state.jump_size, probability_model)
        return jm

    def discounts(self, t: float) -> float:
        """
        Returns the discount factor at time t, based on the jump model.
        """
        if not self.probability_model.is_ready():
            raise ValueError("Probability model not fitted yet.")

        mt = self.state.relevant_meeting_times(0.0, t)
        if mt.size == 0:
            return np.exp(-self.state.r0 * t)

        deltas = np.empty_like(mt)
        deltas[0] = mt[0]
        deltas[1:] = mt[1:] - mt[:-1]

        df = 1.0
        for i, delta in enumerate(deltas):
            p = self.probability_model.probs[i, :]
            f_clean = (np.exp(-self.state.possible_jumps * delta) * p).sum()
            df *= f_clean
        return np.exp(-self.state.r0 * t) * df

    def fwd_rates(self, t: float, T: float) -> float:
        """
        Returns the forward rate between t and T, based on the jump model.
        """
        df_t = self.discounts(t)
        df_T = self.discounts(T)
        return -np.log(df_T / df_t) / (T - t)

    def most_likely_path(self) -> np.ndarray:
        """
        Returns the most likely jump at each segment.
        """
        jumps = []
        for i in range(self.probability_model.probs.shape[0]):
            probs = self.probability_model.probs[i, :]
            j = np.argmax(probs)
            jumps.append(self.state.possible_jumps[j])
        return np.array(jumps) + self.state.r0

    def fit(self, *args, **kwargs):
        return self.probability_model.fit(self.state, *args, **kwargs)

In [8]:
class InfiniteQProbabilityModel(ProbabilityModel):
    """
    A probability models that does not make any assumption on the underlying jump probabilities
    """
    def __init__(self):
        super().__init__()
    
    def gen_probs(self, n_states: int):
        # feasible probabilities
        u = np.random.uniform(0, 1, size=n_states)
        return u/u.sum()

    def discount_per_segment(self, p, delta, state: JumpModelState) -> float:
        dfs = np.exp(-state.possible_jumps * delta)
        return (dfs * p).sum()

    def fit(self, state: JumpModelState, verbose: bool = False):
        clean_dfs, _, end_times, time_deltas = state.curve_implied_meeting_dfs()
        probs = np.empty((end_times.size, state.possible_jumps.size))

        cons = [{'type': 'eq', 'fun': lambda p: np.sum(p) - 1.0}]
        bnds = [(0.0, 1.0)] * state.possible_jumps.shape[0]
        for i, (td, target_df) in enumerate(zip(time_deltas, clean_dfs)):
            z0 = self.gen_probs(state.possible_jumps.shape[0])

            def obj(p):
                err = target_df - self.discount_per_segment(p, td, state)
                return err**2

            res = minimize(obj, z0, bounds=bnds, constraints=cons,
                           options=dict(maxiter=10000, ftol=1e-15))
            probs[i, :] = res.x
            if verbose:
                model = self.discount_per_segment(res.x, td, state)
                print(
                    f"seg {i:02d} | Δ={td:.4f} | target df={target_df:.6f} | model={model:.6f}")

        self.probs = probs
        return probs

In [9]:
model = JumpModel(
    r0=r0,
    curve=curve,
    absolute_meeting_times=absolute_meeting_times,
    jump_size=jump_size,
    possible_jumps=possible_jumps,
    probability_model=InfiniteQProbabilityModel()
)

probs = model.fit(verbose=False)

In [10]:
t = np.arange(0, 10.0, 0.5)
model_dfs = np.array([model.discounts(time) for time in t])
obs_dfs = np.array([curve.discount(time) for time in t])

clean_dfs, _, end_times, deltas = model.state.curve_implied_meeting_dfs()
jumps = model.state.possible_jumps
levels_for_plot = jumps + model.state.r0  # y-axis for the heatmap
model_path = model.most_likely_path()
path_times = model.state.relevant_meeting_times(0.0, t.max())

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.08,
    subplot_titles=("Discount Factors", "Meeting-Outcome Probabilities")
)

fig.add_trace(go.Scatter(x=t, y=model_dfs, mode='lines+markers', name='Model DFs', legend='legend1'), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=obs_dfs,   mode='lines+markers', name='Observed DFs', legend='legend1'), row=1, col=1)

fig.add_trace(
    go.Heatmap(
        x=end_times,
        y=levels_for_plot,
        z=probs.T,
        coloraxis="coloraxis",
        hovertemplate="Meeting: %{x}<br>Level: %{y:.4f}<br>Prob: %{z:.3f}<extra></extra>",
        showscale=True,
        showlegend=False
    ),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(
        x=path_times,
        y=model_path,
        mode='lines+markers',
        name='Most Likely Path',
        line=dict(width=2),
        marker=dict(size=6),
        legend='legend2',
        hoverinfo='skip'
    ),
    row=1, col=2
)

fig.update_layout(
    height=480, width=1100,
    margin=dict(l=70, r=90, t=60, b=110),
    coloraxis=dict(
        colorscale="Viridis",
        colorbar=dict(
            title="Probability",
            x=1.03, xanchor="left",
            y=0.5,  yanchor="middle",
            len=0.8
        )
    ),
    legend=dict(
        orientation="h",
        x=0.25, xanchor="center",
        y=-0.18, yanchor="top",
        title=None
    ),
    legend2=dict(
        orientation="h",
        x=0.75, xanchor="center",
        y=-0.18, yanchor="top",
        title=None
    )
)

# axis titles
fig.update_xaxes(title_text="Time", row=1, col=1)
fig.update_yaxes(title_text="DF",   row=1, col=1)

fig.update_xaxes(title_text="Time", row=1, col=2)
fig.update_yaxes(title_text="Initial Rate + Jumps", row=1, col=2)

fig.update_xaxes(range=[float(end_times.min()), float(end_times.max())], row=1, col=2)

fig.show()

Expert prior: [0.03264418 0.03808487 0.04570185 0.05712731 0.07616975 0.11425462
 0.34276387 0.11425462 0.07616975 0.05712731 0.04570185] sum = 1.0
Number of days: 41


ValueError: could not broadcast input array from shape (48,11) into shape (76,11)

In [22]:
from scipy.special import ndtr

class LatentStateProbabilityModel(ProbabilityModel):

    def __init__(self):
        super().__init__()
        self.x: np.ndarray | None =  None

    def calc_probs(self, x, state: JumpModelState, sigma=None, k=1.0, use_tanh=True, scale=None) -> np.ndarray:
        c = np.sort(state.possible_jumps)
        J = c.size

        edges = np.empty(J + 1)
        edges[1:-1] = 0.5 * (c[1:] + c[:-1])
        edges[0] = -np.inf
        edges[-1] = np.inf

        if sigma is None:
            diffs = np.diff(c)
            sigma = np.median(diffs) if diffs.size else 1.0

        L = max(abs(c[0]), abs(c[-1]))
        x = np.atleast_1d(x)
        if use_tanh:
            mu = L * np.tanh(k * x)
        else:
            if scale is None:
                raise ValueError("Provide 'scale' when use_tanh=False.")
            mu = np.clip(scale * x, c[0], c[-1])

        z_hi = (edges[None, 1:] - mu[:, None]) / sigma
        z_lo = (edges[None, :-1] - mu[:, None]) / sigma
        p = ndtr(z_hi) - ndtr(z_lo)
        return p if p.shape[0] > 1 else p[0]

    def discount_per_segment(self, delta: float, x: float, state: JumpModelState) -> float:
        p = self.calc_probs(x, state)
        dfs = np.exp(-state.possible_jumps * delta)
        return (dfs * p).sum()

    def fit(self, state: JumpModelState, verbose: bool = False):
        clean_obs, _, end_times, deltas = state.curve_implied_meeting_dfs()

        K = end_times.size
        J = state.possible_jumps.size
        probs = np.empty((K, J))
        x = np.empty(K)

        for i, (delta, target) in enumerate(zip(deltas, clean_obs)):
            def obj(x):
                model = self.discount_per_segment(delta, x, state)
                return ((model - target)*100)**2

            res = minimize(obj, x0=np.array([0.0]))
            x[i] = res.x[0]
            probs[i, :] = self.calc_probs(x[i], state)

            if verbose:
                print(
                    f"seg {i:02d} | Δ={delta:.4f} | target={target:.6f} | model={self.discount_per_segment(delta, x[i], state):.6f} | x={x[i]:+.6f}")

        self.probs = probs
        self.x = x
        return probs, x
    


In [23]:
model = JumpModel(
    r0=r0,
    curve=curve,
    absolute_meeting_times=absolute_meeting_times,
    jump_size=jump_size,
    possible_jumps=possible_jumps,
    probability_model=LatentStateProbabilityModel())

probs, x = model.fit(verbose=False)

In [24]:
t = np.arange(0, 10.0, 0.5)
model_dfs = np.array([model.discounts(time) for time in t])
obs_dfs = np.array([curve.discount(time) for time in t])

clean_dfs, _, end_times, deltas = model.state.curve_implied_meeting_dfs()
jumps = model.state.possible_jumps
levels_for_plot = jumps + model.state.r0  # y-axis for the heatmap
model_path = model.most_likely_path()
path_times = model.state.relevant_meeting_times(0.0, t.max())

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.08,
    subplot_titles=("Discount Factors", "Meeting-Outcome Probabilities")
)

# --- Left panel: Discount factors ---
fig.add_trace(go.Scatter(x=t, y=model_dfs, mode='lines+markers',
              name='Model DFs', legend='legend1'), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=obs_dfs,   mode='lines+markers',
              name='Observed DFs', legend='legend1'), row=1, col=1)

fig.add_trace(
    go.Heatmap(
        x=end_times,
        y=levels_for_plot,
        z=probs.T,
        coloraxis="coloraxis",
        hovertemplate="Meeting: %{x}<br>Level: %{y:.4f}<br>Prob: %{z:.3f}<extra></extra>",
        showscale=True,
        showlegend=False
    ),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(
        x=path_times,
        y=model_path,
        mode='lines+markers',
        name='Most Likely Path',
        line=dict(width=2),
        marker=dict(size=6),
        legend='legend2',
        hoverinfo='skip'
    ),
    row=1, col=2
)

fig.update_layout(
    height=480, width=1100,
    margin=dict(l=70, r=90, t=60, b=110),
    coloraxis=dict(
        colorscale="Viridis",
        colorbar=dict(
            title="Probability",
            x=1.03, xanchor="left",
            y=0.5,  yanchor="middle",
            len=0.8
        )
    ),
    legend=dict(
        orientation="h",
        x=0.25, xanchor="center",
        y=-0.18, yanchor="top",
        title=None
    ),
    legend2=dict(
        orientation="h",
        x=0.75, xanchor="center",
        y=-0.18, yanchor="top",
        title=None
    )
)

# axis titles
fig.update_xaxes(title_text="Time", row=1, col=1)
fig.update_yaxes(title_text="DF",   row=1, col=1)

fig.update_xaxes(title_text="Time", row=1, col=2)
fig.update_yaxes(title_text="r0 + jump", row=1, col=2)

fig.update_xaxes(range=[float(end_times.min()),
                 float(end_times.max())], row=1, col=2)

fig.show()

In [30]:
from scipy.optimize import brentq
from scipy.special import logsumexp


class MaxEntropyProbabilityModel(ProbabilityModel):
    def __init__(self, prior: np.ndarray = None):
        super().__init__()
        self.prior = prior
        self.tol: float = 1e-12
        self.lambda_bracket: float = 50.5

    def __expect_a(self, a: np.ndarray, w: np.ndarray, lam: float) -> float:
        z = logsumexp(lam * a, b=w)
        z_a = logsumexp(lam * a, b=w * a)
        return float(np.exp(z_a - z))

    def __solve_lambda(self, a: np.ndarray, D: float, w: np.ndarray) -> float:
        """Solve g(λ)=E[a]-D on active support; assumes D strictly inside."""
        def g(l): return self.__expect_a(a, w, l) - D
        L, U = -self.lambda_bracket, self.lambda_bracket
        gL, gU = g(L), g(U)
        it = 0
        while (gL >= 0 or gU <= 0) and it < 10:
            L *= 2.0
            U *= 2.0
            gL, gU = g(L), g(U)
            it += 1
        return brentq(g, L, U, xtol=self.tol, rtol=self.tol, maxiter=200)

    def fit(self, state: JumpModelState, verbose: bool = False):
        # pull clean per-segment DFs (these are not modified)
        clean_dfs, _, end_times, deltas = state.curve_implied_meeting_dfs()
        K = end_times.size
        J = state.possible_jumps.size

        # build per-segment prior weights W (no alteration of user input)
        if self.prior is None:
            W = np.ones((K, J)) / J
        else:
            P = np.asarray(self.prior, dtype=float)
            if P.ndim == 1:
                if P.shape[0] != J:
                    raise ValueError(f"prior length {P.shape[0]} != J={J}")
                W = np.tile(P, (K, 1))
            elif P.ndim == 2:
                if P.shape != (K, J):
                    raise ValueError(
                        f"prior shape {P.shape} != (K={K}, J={J})")
                W = P
            else:
                raise ValueError("prior must be 1D (J,) or 2D (K,J)")
            # normalize rows; if a row sums to zero, that's an invalid prior
            row_sums = W.sum(axis=1, keepdims=True)
            if np.any(row_sums <= 0):
                bad = np.where(row_sums.ravel() <= 0)[0]
                raise ValueError(
                    f"prior row(s) {bad.tolist()} sum to zero; cannot define active support.")
            W = W / row_sums

        probs = np.empty((K, J))

        for k, (delta, Dk) in enumerate(zip(deltas, clean_dfs)):
            a = np.exp(-state.possible_jumps * delta)   # (J,)
            w = W[k]                                    # (J,)

            # active support given the prior (strict — no changes to w)
            mask = w > 0
            if not np.any(mask):
                raise ValueError(
                    f"segment {k}: prior has no positive weights; infeasible.")
            a_active = a[mask]
            amin, amax = float(a_active.min()), float(a_active.max())

            # Feasibility strictly on active support (no projection of Dk)
            if Dk < amin - self.tol or Dk > amax + self.tol:
                raise ValueError(
                    f"segment {k}: D={Dk:.12f} not in active support [{amin:.12f},{amax:.12f}]. "
                    "Either widen possible_jumps, adjust r0/meetings, or provide a prior with support there."
                )

            # Boundary → degenerate at argmin/argmax within active support
            if abs(Dk - amin) <= self.tol:
                pk = np.zeros(J)
                idx = np.argmin(np.where(mask, a, np.inf))
                pk[idx] = 1.0
            elif abs(Dk - amax) <= self.tol:
                pk = np.zeros(J)
                idx = np.argmax(np.where(mask, a, -np.inf))
                pk[idx] = 1.0
            else:
                # Strictly inside → unique finite lambda
                lam = self.__solve_lambda(a, Dk, w)
                logZ = logsumexp(lam * a, b=w)
                pk = np.exp(lam * a - logZ) * w

            # normalize (no clipping)
            s = pk.sum()
            if s <= 0:
                raise RuntimeError(
                    f"segment {k}: degenerate probability vector.")
            pk /= s
            probs[k] = pk

            if verbose:
                D_model = float(np.dot(pk, a))
                print(
                    f"seg {k:02d} | Δ={delta:.6f} | target={Dk:.12f} | model={D_model:.12f}")

        self.probs = probs
        return probs

In [31]:
model = JumpModel(
    r0=r0,
    curve=curve,
    absolute_meeting_times=absolute_meeting_times,
    jump_size=jump_size,
    possible_jumps=possible_jumps,
    probability_model=MaxEntropyProbabilityModel(prior=None))

probs = model.fit()

In [32]:
t = np.arange(0, 10.0, 0.5)
model_dfs = np.array([model.discounts(time) for time in t])
obs_dfs = np.array([curve.discount(time) for time in t])

clean_dfs, _, end_times, deltas = model.state.curve_implied_meeting_dfs()
jumps = model.state.possible_jumps
levels_for_plot = jumps + model.state.r0  # y-axis for the heatmap
model_path = model.most_likely_path()
path_times = model.state.relevant_meeting_times(0.0, t.max())

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.08,
    subplot_titles=("Discount Factors", "Meeting-Outcome Probabilities")
)

# --- Left panel: Discount factors ---
fig.add_trace(go.Scatter(x=t, y=model_dfs, mode='lines+markers',
              name='Model DFs', legend='legend1'), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=obs_dfs,   mode='lines+markers',
              name='Observed DFs', legend='legend1'), row=1, col=1)

fig.add_trace(
    go.Heatmap(
        x=end_times,
        y=levels_for_plot,
        z=probs.T,
        coloraxis="coloraxis",
        hovertemplate="Meeting: %{x}<br>Level: %{y:.4f}<br>Prob: %{z:.3f}<extra></extra>",
        showscale=True,
        showlegend=False
    ),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(
        x=path_times,
        y=model_path,
        mode='lines+markers',
        name='Most Likely Path',
        line=dict(width=2),
        marker=dict(size=6),
        legend='legend2',
        hoverinfo='skip'
    ),
    row=1, col=2
)

fig.update_layout(
    height=480, width=1100,
    margin=dict(l=70, r=90, t=60, b=110),
    coloraxis=dict(
        colorscale="Viridis",
        colorbar=dict(
            title="Probability",
            x=1.03, xanchor="left",
            y=0.5,  yanchor="middle",
            len=0.8
        )
    ),
    legend=dict(
        orientation="h",
        x=0.25, xanchor="center",
        y=-0.18, yanchor="top",
        title=None
    ),
    legend2=dict(
        orientation="h",
        x=0.75, xanchor="center",
        y=-0.18, yanchor="top",
        title=None
    )
)

# axis titles
fig.update_xaxes(title_text="Time", row=1, col=1)
fig.update_yaxes(title_text="DF",   row=1, col=1)
fig.update_xaxes(title_text="Time", row=1, col=2)
fig.update_yaxes(title_text="r0 + jump", row=1, col=2)
fig.update_xaxes(range=[float(end_times.min()),
                 float(end_times.max())], row=1, col=2)
fig.show()

In [51]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize

# ============================================================
# 1) Expert prior over jumps
# ============================================================

def make_expert_prior(possible_jumps: np.ndarray) -> np.ndarray:
    """
    Example expert prior:
    - highest probability at 0 jump (no change),
    - symmetric decay with |jump| in bp.
    Replace with your actual expert view if needed.
    """
    jumps_bps = possible_jumps * 10_000
    scale = 1.0 / (1.0 + np.abs(jumps_bps) / 25.0)
    zero_mask = np.isclose(jumps_bps, 0.0)
    scale[zero_mask] *= 1.5
    q = scale / scale.sum()
    return q

expert_prior = make_expert_prior(possible_jumps)
# If you have a specific vector, you can override:
# expert_prior = np.array([...], dtype=float); expert_prior /= expert_prior.sum()


# ============================================================
# 2) Build JumpModelState for each curve date (same visibility)
# ============================================================

def make_state_for_date(date, meetings_visibility: float = 5.0) -> JumpModelState:
    curve_d = Curve(
        tenors=np.asarray(discounts.columns, dtype=float),
        dfs=discounts.loc[date].to_numpy(dtype=float),
    )
    r0_d = (1.0 / curve_d.discount(dt) - 1.0) / dt

    return JumpModelState(
        r0=r0_d,
        curve=curve_d,
        absolute_meeting_times=absolute_meeting_times,
        possible_jumps=possible_jumps,
        jump_size=jump_size,
        meetings_visibility=meetings_visibility,
    )

# choose a window of dates around some reference index
center_idx = 1000
window_radius = 20   # ~41 days; adjust as you like
window = discounts.index[center_idx-window_radius:center_idx+window_radius+1]
states = [make_state_for_date(d, meetings_visibility=5.0) for d in window]
D = len(states)
print("Number of days:", D)


# ============================================================
# 3) Panel fit: day 0 ~ expert, others ~ previous day
#    - Softmax parameterization (no constraints)
#    - Strong weight on curve-fit term to get good fit
# ============================================================

def panel_fit_firstday_prior_then_smoothing(
    states: list[JumpModelState],
    expert_prior: np.ndarray,
    lam_prior: float = 1.0,
    lam_smooth: float = 5.0,
    data_weight: float = 1.0,
    verbose: bool = False,
):
    """
    Joint panel calibration, per segment k:

        min_{p_d}  sum_d  [ data_weight * ( (E_p_d[dfs_j] - D_{d,k}) in bps )^2 ]
                  + lam_prior * ||p_0 - q||^2
                  + lam_smooth * sum_{d>0} ||p_d - p_{d-1}||^2

    where:
      - q = expert_prior (length J),
      - p_d are probability vectors over jumps for day d, segment k,
      - D_{d,k} = clean per-segment DF from curve.

    Simplex constraints enforced via softmax logits.

    Returns:
        panel_probs: (D, K_common, J)
        end_times_common: (K_common,)
        deltas_common: (K_common,)
    """
    D = len(states)
    if D == 0:
        raise ValueError("No states provided.")

    # --- collect per-day clean DFs and segment grids ---
    clean_dfs_list = []
    deltas_list = []
    end_times_list = []
    K_list = []

    for state in states:
        clean_dfs, _, end_times, deltas = state.curve_implied_meeting_dfs()
        clean_dfs_list.append(clean_dfs)   # (K_d,)
        deltas_list.append(deltas)         # (K_d,)
        end_times_list.append(end_times)   # (K_d,)
        K_list.append(clean_dfs.size)

    # Use only segments that exist for all days
    K_common = min(K_list)
    if verbose:
        print(f"Segments per day: min={K_common}, max={max(K_list)}. Using first K_common segments.")

    jumps = states[0].possible_jumps
    J = jumps.size

    # Normalize expert prior q
    q = np.asarray(expert_prior, dtype=float).ravel()
    if q.size != J:
        raise ValueError(f"expert_prior length {q.size} != number of jumps {J}")
    if q.sum() <= 0:
        raise ValueError("expert_prior must have positive sum.")
    q /= q.sum()

    # Stack clean DFs and deltas for first K_common segments
    clean_dfs_panel = np.stack([c[:K_common] for c in clean_dfs_list], axis=0)  # (D, K_common)
    deltas_panel    = np.stack([d[:K_common] for d in deltas_list], axis=0)     # (D, K_common)

    end_times_common = end_times_list[0][:K_common]
    deltas_common    = deltas_panel[0, :]

    panel_probs = np.empty((D, K_common, J))

    def softmax_rows(Z):
        Z_shift = Z - Z.max(axis=1, keepdims=True)
        eZ = np.exp(Z_shift)
        return eZ / eZ.sum(axis=1, keepdims=True)  # (D, J)

    # --- per-segment optimisation ---
    for k in range(K_common):
        targets_k = clean_dfs_panel[:, k]   # (D,)
        deltas_k  = deltas_panel[:, k]      # (D,)

        def unpack(Z_flat):
            return Z_flat.reshape(D, J)

        def obj(Z_flat):
            Z = unpack(Z_flat)
            P = softmax_rows(Z)  # (D, J)

            # data term: fit per-day D_{d,k}
            data_term = 0.0
            for d_idx in range(D):
                dfs_j = np.exp(-jumps * deltas_k[d_idx])  # (J,)
                model_df = float(np.dot(P[d_idx], dfs_j))
                # scale DF error into bps to give it "size"
                err_bps = (model_df - targets_k[d_idx]) * 10_000.0
                data_term += data_weight * (err_bps * err_bps)

            # prior term: ONLY on day 0
            prior_term = 0.0
            if lam_prior > 0.0:
                diff0 = P[0] - q
                prior_term = lam_prior * float(np.dot(diff0, diff0))

            # temporal smoothing: day d>0 close to day d-1
            smooth_term = 0.0
            if lam_smooth > 0.0 and D > 1:
                diffP = P[1:, :] - P[:-1, :]
                smooth_term = lam_smooth * np.sum(diffP * diffP)

            return data_term + prior_term + smooth_term

        # Initial logits: all days start from expert prior
        Z0 = np.log(q + 1e-12)[None, :].repeat(D, axis=0)   # (D, J)
        z0_flat = Z0.ravel()

        res = minimize(
            obj,
            z0_flat,
            method="L-BFGS-B",
            options=dict(maxiter=600, ftol=1e-9),
        )

        if not res.success and verbose:
            print(f"Segment {k}: optimisation failed: {res.message}")

        Z_opt = unpack(res.x)
        P_opt = softmax_rows(Z_opt)
        panel_probs[:, k, :] = P_opt

        if verbose:
            model_df_vec = np.array([
                np.dot(P_opt[d_idx], np.exp(-jumps * deltas_k[d_idx]))
                for d_idx in range(D)
            ])
            err_bps = (model_df_vec - targets_k) * 10_000.0
            rms_bps = np.sqrt(np.mean(err_bps**2))
            print(f"seg {k:02d} | RMS DF error ≈ {rms_bps:.4f} bps")

    return panel_probs, end_times_common, deltas_common


# ---- run calibration ----
lam_prior  = 1.0   # first day near expert
lam_smooth = 1.0   # closeness to previous day
data_weight = 1.0  # DF fit weight; can increase if needed

panel_probs, end_times, deltas_common = panel_fit_firstday_prior_then_smoothing(
    states,
    expert_prior=expert_prior,
    lam_prior=lam_prior,
    lam_smooth=lam_smooth,
    data_weight=data_weight,
    verbose=False,
)

D, K_common, J = panel_probs.shape
print("panel_probs shape:", panel_probs.shape)


# ============================================================
# 4) Evolution of probabilities for the *next* meeting
# ============================================================

dates = np.array(window)[:D]
front_probs = panel_probs[:, 0, :]   # (D, J)

step_bps = int(round(jump_size * 10_000))
idx = (np.arange(-(J//2), J//2 + 1) if J % 2 else np.arange(-J//2, J//2))
jumps_bps_grid = idx * step_bps

mask_down = jumps_bps_grid < 0
mask_stay = np.isclose(jumps_bps_grid, 0)
mask_up   = jumps_bps_grid > 0

series_down = front_probs[:, mask_down].sum(axis=1) if mask_down.any() else np.zeros(D)
series_stay = front_probs[:, mask_stay].sum(axis=1) if mask_stay.any() else np.zeros(D)
series_up   = front_probs[:, mask_up].sum(axis=1)   if mask_up.any()   else np.zeros(D)

total = series_down + series_stay + series_up
total[total == 0] = 1.0
series_down /= total
series_stay /= total
series_up   /= total

fig_prob_evol = go.Figure()
fig_prob_evol.add_trace(go.Scatter(x=dates, y=series_down, name="Down (<0)", mode="lines+markers"))
fig_prob_evol.add_trace(go.Scatter(x=dates, y=series_stay, name="Stay (0)",  mode="lines+markers"))
fig_prob_evol.add_trace(go.Scatter(x=dates, y=series_up,   name="Up (>0)",   mode="lines+markers"))

fig_prob_evol.update_layout(
    title="Evolution of Probabilities for Next Meeting\n(Day 0 ~ Expert, Others ~ Previous Day)",
    xaxis_title="Date",
    yaxis_title="Probability",
    yaxis=dict(tickformat=".0%", range=[0, 1]),
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="left", x=0),
    height=500,
    margin=dict(l=60, r=40, t=60, b=80),
)

fig_prob_evol.show()


# ============================================================
# 5) For one date: discounts, probabilities, forward rates
# ============================================================

# choose a date index to inspect (e.g. middle of the window)
d_idx = D // 2
state_d = states[d_idx]

jm_d = JumpModel(
    r0=state_d.r0,
    curve=state_d.curve,
    absolute_meeting_times=state_d.absolute_meeting_times,
    possible_jumps=state_d.possible_jumps,
    jump_size=state_d.jump_size,
    probability_model=InfiniteQProbabilityModel(),  # container for probs
)
jm_d.probability_model.probs = panel_probs[d_idx, :, :]  # (K_common, J)

# Limit time grid to the calibration horizon to avoid asking for
# more segments than we have probabilities for
t_max = float(end_times.max())
t = np.arange(0, t_max + 0.5, 0.5)

# Discount factors: model vs observed
model_dfs = np.array([jm_d.discounts(time) for time in t])
obs_dfs   = np.array([state_d.curve.discount(time) for time in t])

# Forward rates: model vs curve
fwd_model = np.array([jm_d.fwd_rates(tt, tt + dt) for tt in t])
fwd_obs   = np.array([state_d.curve.fwd_rate(tt, tt + dt) for tt in t])

# Heatmap info
jumps = state_d.possible_jumps
levels_for_plot = jumps + state_d.r0
model_path = jm_d.most_likely_path()
path_times = end_times  # we calibrated exactly on these segments

fig = make_subplots(
    rows=2, cols=2,
    horizontal_spacing=0.08,
    vertical_spacing=0.12,
    subplot_titles=(
        "Discount Factors (Panel-Calibrated)",
        "Meeting-Outcome Probabilities",
        "Forward Rates",
        ""
    ),
)

# (1,1) Discount factors
fig.add_trace(go.Scatter(x=t, y=model_dfs, mode='lines+markers', name='Model DFs'), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=obs_dfs,   mode='lines+markers', name='Observed DFs'), row=1, col=1)

# (1,2) Probabilities heatmap + most likely path
fig.add_trace(
    go.Heatmap(
        x=end_times,
        y=levels_for_plot,
        z=panel_probs[d_idx].T,
        coloraxis="coloraxis",
        hovertemplate="Meeting: %{x}<br>Level: %{y:.4f}<br>Prob: %{z:.3f}<extra></extra>",
        showscale=True,
        showlegend=False
    ),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(
        x=path_times,
        y=model_path,
        mode='lines+markers',
        name='Most Likely Path',
        line=dict(width=2),
        marker=dict(size=6),
        hoverinfo='skip'
    ),
    row=1, col=2
)

# (2,1) Forward rates
fig.add_trace(
    go.Scatter(x=t, y=fwd_model * 100.0, mode='lines+markers', name='Model Fwds'),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=t, y=fwd_obs * 100.0, mode='lines+markers', name='Observed Fwds'),
    row=2, col=1
)

fig.update_layout(
    height=800, width=1100,
    margin=dict(l=70, r=90, t=60, b=110),
    coloraxis=dict(
        colorscale="Viridis",
        colorbar=dict(
            title="Probability",
            x=1.03, xanchor="left",
            y=0.5,  yanchor="middle",
            len=0.8
        )
    ),
    legend=dict(
        orientation="h",
        x=0.5, xanchor="center",
        y=-0.12, yanchor="top",
        title=None
    )
)

fig.update_xaxes(title_text="Time", row=1, col=1)
fig.update_yaxes(title_text="DF",   row=1, col=1)
fig.update_xaxes(title_text="Time", row=1, col=2)
fig.update_yaxes(title_text="r0 + jump", row=1, col=2)
fig.update_xaxes(title_text="Time", row=2, col=1)
fig.update_yaxes(title_text="Fwd Rate (%)", row=2, col=1)

fig.update_xaxes(
    range=[float(end_times.min()), float(end_times.max())],
    row=1, col=2
)

fig.show()


Number of days: 41
panel_probs shape: (41, 76, 11)
